In [ ]:
from pathlib import Path

import boto3

import duckdb

from tfl_ttl import config
from tfl_ttl.s3 import download_prefix

# From .env, so the bucket name (which contains the AWS account ID) isn't in
# this public repo.
BUCKET = config.S3_BUCKET

# Fail loudly rather than download into the wrong folder: every path below is
# built relative to where Jupyter was started.
if Path.cwd().name != "notebooks":
	raise ValueError("Run this notebook from the repo's notebooks folder.")

ROOT = Path.cwd().parent
# data/raw is gitignored, so downloaded polls can never be committed by accident.
LOCAL_RAW = ROOT / "data" / "raw"


# A database FILE instead of memory, so views survive kernel restarts.
# data/processed/ is gitignored, so it can't be committed by accident.
PROFILING_DB = ROOT / "data" / "processed" / "profiling.duckdb"
# Every duckdb.sql(...) in this notebook (and the DuckDB UI started from it)
# now reads and writes this file instead of a throwaway in-memory database.
duckdb.set_default_connection(duckdb.connect(str(PROFILING_DB)))

# Start the DuckDB UI in a browser tab. It connects to the same database file, so you can explore the tables and views created by this notebook.
duckdb.sql("CALL start_ui()")

# The read-only identity: an explicit profile ignores the writer keys in .env.
reader = boto3.Session(profile_name="tfl-reader").client("s3")

# Check modes

In [ ]:
# Returns the files downloaded this time. [] means everything was already here:
# raw files never change, so existing copies are skipped, not re-fetched.
download_prefix(reader, BUCKET, "raw/tfl_verify/modes", LOCAL_RAW)

import duckdb

# Every downloaded modes file. ** means "any folders in between", so this
# matches every poll_date=... folder. as_posix() gives forward slashes, which
# DuckDB needs even on Windows.
modes_files = (LOCAL_RAW / "raw/tfl_verify/modes/**/*.json.gz").as_posix()

duckdb.sql(f"""
    -- One row per poll: just the wrapper the pipeline adds around TfL's response.
    SELECT
        feed,                               -- which feed this poll was
        mode,                               -- NULL for feeds without modes
        polled_at,                          -- when the request was made (UTC)
        poll_date,                          -- not in the file: read from the poll_date=... folder name
        record_count,                       -- what the pipeline counted when it wrote the file
        len(response) AS records_in_file    -- what's actually in the file, counted now
                                            -- (len() of a list = number of items in it)

    -- read_json reads JSON straight from files, one row per line. It spots
    -- the .gz and decompresses it, and works out column types from the data.
    FROM read_json(
        '{modes_files}',
        -- Turn key=value folder names (poll_date=2026-10-02) into columns.
        -- The same trick Athena uses to skip folders it doesn't need.
        hive_partitioning = true
    )
""")

In [ ]:
import duckdb

duckdb.sql(f"""
   -- Column profile: one row of stats per field inside the response.
   SELECT *
   FROM
   (
   -- SUMMARIZE in front of a SELECT returns stats about its columns instead
   -- of its rows: type, min, max, approx_unique, count, null_percentage...
   SUMMARIZE SELECT
        -- unnest spreads the response list into one row per item; with
        -- recursive := true each field becomes its own column.
        unnest(response, recursive := true)   -- 18 rows x 5 columns, one per field
    FROM read_json('{modes_files}')

    )
    -- SUMMARIZE's output is a normal result set, so it can be wrapped in a
    -- subquery and filtered like any table. Remove this WHERE to see the
    -- stats (e.g. null %) for all 5 fields.
    WHERE column_name = 'modeName'
""")
# approx_unique is an estimate: it reads 21 here for 18 real values. Fine for
# spotting "mostly unique" vs "few values", but use count(DISTINCT ...) for an
# exact key check, as in the cell below.

In [ ]:
duckdb.sql(f"""
   -- Every distinct mode name: eyeball the list against what TfL documents.
   SELECT DISTINCT modeName
   FROM
   (
   -- The same unnest as above: one row per mode, one column per field.
   -- Wrapping it in brackets makes it a subquery: a temporary table the outer
   -- SELECT reads from.
   SELECT
        unnest(response, recursive := true)   -- 18 rows x 5 columns, one per field
    FROM read_json('{modes_files}')
    )
""")

In [ ]:
duckdb.sql(f"""
    -- Key check: can modeName identify a row in a future dim_mode table?
    -- It can only if every row has a different value, i.e. the two counts match.
    SELECT
    count(*)                 AS total_rows,      -- every row
    count(DISTINCT modeName) AS distinct_names   -- each value counted once (exact)
    FROM    (
   SELECT
        unnest(response, recursive := true)   -- 18 rows x 5 columns, one per field
    FROM read_json('{modes_files}')
    )
""")
# 18 = 18, so modeName is unique within one poll. It only proves uniqueness in
# this one file: with many polls stacked together, the key becomes
# (poll, modeName), and each mode repeats once per poll.

In [ ]:
duckdb.sql(f"""
    -- Name the unnest once, instead of repeating the subquery in every cell.
    -- A view stores the query, not the data: each time it's used, DuckDB re-runs
    -- it against the files. OR REPLACE lets this cell be re-run without an
    -- "already exists" error.
    CREATE OR REPLACE VIEW stg_modes AS
    SELECT
        -- Keep the poll's identity on every row. Within one file modeName is
        -- unique, but across many polls the grain is (poll, modeName).
        polled_at,
        poll_date,                              -- from the poll_date=... folder
        unnest(response, recursive := true)     -- one row per mode, one column per field
    FROM read_json('{modes_files}', hive_partitioning = true)
""")
# This is the shape of a dbt staging model: one place that turns raw JSON into
# typed rows, which everything downstream selects from. In dbt it would be a
# file called stg_modes.sql. The view lives in DuckDB's default in-memory
# database, so it lasts until the notebook's kernel restarts.

# The earlier queries, now reading from the view:
duckdb.sql("""
    -- The full profile for every column, no WHERE this time.
    -- SUMMARIZE can take a table or view name directly.
    SELECT column_name, column_type, count, null_percentage, approx_unique
    FROM (SUMMARIZE stg_modes)
""").show()

duckdb.sql("""
    -- The key check, per poll: the two counts should match.
    SELECT
        count(*)                                     AS total_rows,
        count(DISTINCT (polled_at, modeName))        AS distinct_poll_mode_pairs
    FROM stg_modes
""").show()
# .show() prints a result mid-cell. Without it, a notebook only displays the
# last expression of the cell.

# Checking Severity

In [ ]:
download_prefix(reader, BUCKET, "raw/tfl_verify/severity", LOCAL_RAW)

In [ ]:
severity_files = (LOCAL_RAW / "raw/tfl_verify/severity/**/*.json.gz").as_posix()

duckdb.sql(f"""
    -- One row per poll: just the wrapper the pipeline adds around TfL's response.
    SELECT
        feed,                               -- which feed this poll was
        mode,                               -- NULL for feeds without modes
        polled_at,                          -- when the request was made (UTC)
        poll_date,                          -- not in the file: read from the poll_date=... folder name
        record_count,                       -- what the pipeline counted when it wrote the file
        len(response) AS records_in_file    -- what's actually in the file, counted now
    FROM read_json(
        '{severity_files}',
        hive_partitioning = true
    )
""")

In [ ]:
duckdb.sql(f"""
    CREATE OR REPLACE VIEW stg_severity AS
    SELECT
        polled_at,
        poll_date,                              -- from the poll_date=... folder
        unnest(response, recursive := true)     -- one row per mode, one column per field
    FROM read_json('{severity_files}', hive_partitioning = true)
""")


# The earlier queries, now reading from the view:
duckdb.sql("""
    -- The full profile for every column, no WHERE this time.
    -- SUMMARIZE can take a table or view name directly.
    SELECT column_name, column_type, count, null_percentage, approx_unique
    FROM (SUMMARIZE stg_severity)
""").show()

duckdb.sql("""
    SELECT modeName, severityLevel, description
    FROM stg_severity
""").show()
# .show() prints a result mid-cell. Without it, a notebook only displays the
# last expression of the cell.



# Check Disruption Categories

In [ ]:
# --- disruption_categories ---
# Returns [] if the file is already here (I downloaded it for you just now).
download_prefix(reader, BUCKET, "raw/tfl_verify/disruption_categories", LOCAL_RAW)

categories_files = (
    LOCAL_RAW / "raw/tfl_verify/disruption_categories/**/*.json.gz"
).as_posix()

duckdb.sql(f"""
    CREATE OR REPLACE VIEW stg_disruption_categories AS
    SELECT
        polled_at,                  -- the poll's identity, on every row
        poll_date,                  -- from the poll_date=... folder
        unnest(response) AS category
            -- This response is a list of plain text values, not records,
            -- so there are no fields to spread out: no recursive := true.
            -- AS category names the single resulting column.
    FROM read_json('{categories_files}', hive_partitioning = true)
""")

# Check Lines

In [ ]:
# --- lines ---
# Returns [] if the files are already here (I downloaded them for you just now).
download_prefix(reader, BUCKET, "raw/tfl_verify/lines", LOCAL_RAW)

lines_files = (LOCAL_RAW / "raw/tfl_verify/lines/**/*.json.gz").as_posix()

duckdb.sql(f"""
    CREATE OR REPLACE VIEW stg_lines AS
    SELECT
        polled_at,
        poll_date,
        -- The wrapper's mode (which call this came from). Renamed so it can't be
        -- confused with modeName, which is inside TfL's response.
        mode AS polled_mode,
        unnest(response, recursive := true)  -- one row per line, one column per field
    -- The ** pattern now matches 4 files (one per mode). read_json reads them
    -- all and stacks their rows into one result.
    FROM read_json('{lines_files}', hive_partitioning = true)
""")

# Check Line Status

In [ ]:
# --- line_status ---
# Returns [] if the files are already here (I downloaded them for you just now).
download_prefix(reader, BUCKET, "raw/tfl_verify/line_status", LOCAL_RAW)

line_status_files = (LOCAL_RAW / "raw/tfl_verify/line_status/**/*.json.gz").as_posix()

duckdb.sql(f"""
    CREATE OR REPLACE VIEW stg_line_status AS

    -- Nested twice: the response is a list of lines, and each line has its own
    -- list of statuses. So unnest twice.

    -- WITH ... AS (...) is a CTE: a named step used by the query below it.
    -- Step 1: one row per line.
    WITH lines AS (
        SELECT
            polled_at,
            poll_date,
            mode AS polled_mode,          -- which call this came from
            unnest(response) AS line      -- no recursive: keep each line as one struct
        FROM read_json('{line_status_files}', hive_partitioning = true)
    )

    -- Step 2: one row per status on each line. This is fact_line_status's grain.
    SELECT
        polled_at,
        poll_date,
        polled_mode,
        line.id        AS line_id,        -- struct.field reads one field from a struct
        line.name      AS line_name,
        line.modeName  AS mode_name,

        -- Every status id is 0, so position in the list is what tells two
        -- statuses on one line apart. generate_subscripts numbers the list's
        -- items 1, 2, 3... and DuckDB keeps it in step with the unnest below.
        generate_subscripts(line.lineStatuses, 1) AS status_position,
        unnest(line.lineStatuses)                 AS status   -- one row per status

    FROM lines
""")

# Check Line Disruptions

In [ ]:
# --- stop_point_disruptions ---
# Returns [] if the files are already here (I downloaded them for you just now).
download_prefix(reader, BUCKET, "raw/tfl_verify/stop_point_disruptions", LOCAL_RAW)

spd_files = (
    LOCAL_RAW / "raw/tfl_verify/stop_point_disruptions/**/*.json.gz"
).as_posix()

duckdb.sql(f"""
    CREATE OR REPLACE VIEW stg_stop_point_disruptions AS
    WITH points AS (
        SELECT
            polled_at,
            poll_date,
            mode AS polled_mode,         -- the wrapper's mode. TfL's records ALSO have a
                                         -- field called "mode", so renaming avoids a clash
            unnest(response) AS d        -- one row per disrupted stop point
        FROM read_json('{spd_files}', hive_partitioning = true)
    )
    SELECT
        polled_at,
        poll_date,
        polled_mode,
        d.stationAtcoCode  AS station_code,   -- the station it belongs to (never null here)
        d.atcoCode         AS stop_code,      -- the exact point; same as station_code in this poll
        d.commonName       AS station_name,
        d.type             AS disruption_type,     -- Information / Interchange Message / Closure...
        d.appearance       AS appearance,          -- RealTime / PlannedWork / Information
        d.fromDate         AS from_date,
        d.toDate           AS to_date,
        d.description      AS description,
        d.concernedLines   AS concerned_lines      -- a list of lines; null for ~96% of rows
    FROM points
""")

# Check Stop Points

In [ ]:
# --- stop_points ---
download_prefix(reader, BUCKET, "raw/tfl_verify/stop_points", LOCAL_RAW)

sp_files = (LOCAL_RAW / "raw/tfl_verify/stop_points/**/*.json.gz").as_posix()

duckdb.sql(f"""
    CREATE OR REPLACE VIEW stg_stop_points AS
    WITH points AS (
        SELECT
            polled_at,
            poll_date,
            mode AS polled_mode,
            -- The list is inside the response object, so reach in with a dot.
            unnest(response.stopPoints) AS s
        FROM read_json(
            '{sp_files}',
            hive_partitioning = true,
            -- Allow single JSON objects up to 50 MB; one tube poll is one big object.
            maximum_object_size = 50000000
        )
    )
    SELECT
        polled_at, poll_date, polled_mode,
        s.naptanId       AS stop_code,       -- this exact point (station, platform, entrance...)
        s.stationNaptan  AS station_code,    -- the station it belongs to (NULL for hubs)
        s.stopType       AS stop_type,       -- NaptanMetroStation, NaptanMetroPlatform, ...
        s.commonName     AS stop_name,
        s.lat, s.lon,
        list_transform(s.lines, x -> x.id) AS line_ids   -- just the line ids from each line struct
    FROM points
""")